# 01 Probability sampling
**Chapter 3: Methodology — D2 Probability sampling, worked examples.** The Python twin of `01_sampling.do`.

Frame: `sme_sampling_frame.csv`, a hypothetical list of 2,400 SMEs. We draw samples of n = 331,
the Krejcie–Morgan size for N = 2,400. Random draws differ from Stata's (different generators),
but every allocation and every design rule is the same.

In [1]:
# Setup: install anything missing (needed on Google Colab), then load the libraries.
import importlib.util, subprocess, sys
for pkg in ["pandas", "numpy", "scipy", "statsmodels", "linearmodels", "matplotlib", "openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Data live in ../data next to this notebook; online (e.g. Colab) they are read from GitHub.
DATA = Path("../data")
RAW_URL = "https://raw.githubusercontent.com/soonjanjan-lang/Chapter3-Methodology-lectnotes-dashboard/main/downloads/data/"

def load(name):
    p = DATA / name
    return pd.read_csv(p) if p.exists() else pd.read_csv(RAW_URL + name)

In [2]:
frame = load("sme_sampling_frame.csv")
N, n = len(frame), 331
rng = np.random.default_rng(20261019)     # report the seed in your thesis
print("N =", N)
frame["sector"].value_counts()

N = 2400


sector
Services         1560
Manufacturing     480
Agriculture       240
Construction      120
Name: count, dtype: int64

## A. Simple random sampling (without replacement)

In [3]:
srs = frame.sample(n=n, random_state=20261019)
print(len(srs))
pd.concat([frame["sector"].value_counts(normalize=True).rename("frame"),
           srs["sector"].value_counts(normalize=True).rename("SRS")], axis=1).round(3)

331


,frame,SRS
sector,,
Services,0.65,0.674
Manufacturing,0.20,0.190
Agriculture,0.10,0.094
Construction,0.05,0.042


## B. Systematic random sampling
Interval k = floor(N / n); random start between 1 and k; then every k-th unit of the sorted frame.

In [4]:
k = N // n
start = int(rng.integers(1, k + 1))
ordered = frame.sort_values("sme_id").reset_index(drop=True)
sys_sample = ordered.iloc[start - 1::k].head(n)          # k = 7 gives 342 or 343; keep the first 331
print(f"k = {k}, random start = {start}, n = {len(sys_sample)}")
sys_sample["sector"].value_counts()

k = 7, random start = 3, n = 331


sector
Services         212
Manufacturing     77
Agriculture       28
Construction      14
Name: count, dtype: int64

## C. Stratified random sampling: proportionate allocation, n_h = n × N_h / N

In [5]:
alloc = frame.groupby("sector").size().rename("N_h").to_frame()
alloc["n_prop"] = (n * alloc["N_h"] / N).round().astype(int)
print(alloc, "\nTotal:", alloc["n_prop"].sum())
prop = pd.concat([g.sample(n=alloc.loc[s, "n_prop"], random_state=1) for s, g in frame.groupby("sector")])
prop["sector"].value_counts()

                N_h  n_prop
sector                     
Agriculture     240      33
Construction    120      17
Manufacturing   480      66
Services       1560     215 
Total: 331


sector
Services         215
Manufacturing     66
Agriculture       33
Construction      17
Name: count, dtype: int64

## D. Disproportionate allocation, with design weights w_h = N_h / n_h

In [6]:
alloc["n_dis"] = pd.Series({"Services": 141, "Manufacturing": 70, "Agriculture": 60, "Construction": 60})
alloc["w"] = alloc["N_h"] / alloc["n_dis"]
print(alloc)
dis = pd.concat([g.sample(n=alloc.loc[s, "n_dis"], random_state=2) for s, g in frame.groupby("sector")])
dis["w"] = dis["sector"].map(alloc["w"])
print("Population share accepting e-wallets:", round(frame["uses_ewallet_payments"].mean(), 4))
print("Unweighted sample estimate:          ", round(dis["uses_ewallet_payments"].mean(), 4))
print("Weighted sample estimate:            ", round(np.average(dis["uses_ewallet_payments"], weights=dis["w"]), 4))

                N_h  n_prop  n_dis          w
sector                                       
Agriculture     240      33     60   4.000000
Construction    120      17     60   2.000000
Manufacturing   480      66     70   6.857143
Services       1560     215    141  11.063830
Population share accepting e-wallets: 0.4567
Unweighted sample estimate:           0.3505
Weighted sample estimate:             0.4233


## E. Sekaran & Bougie (2016, Table 13.1): proportionate (20%) and disproportionate samples

In [7]:
sek = pd.DataFrame({
    "job_level": ["Top management", "Middle-level management", "Lower-level management", "Supervisors", "Clerks", "Secretaries"],
    "N_h": [10, 30, 50, 100, 500, 20],
    "n_disproportionate": [7, 15, 20, 30, 60, 10]})
sek["n_proportionate"] = (0.20 * sek["N_h"]).astype(int)
sek.loc["Total"] = ["Total", sek["N_h"].sum(), sek["n_disproportionate"].sum(), sek["n_proportionate"].sum()]
sek

,job_level,N_h,n_disproportionate,n_proportionate
0,Top management,10,7,2
1,Middle-level management,30,15,6
2,Lower-level management,50,20,10
3,Supervisors,100,30,20
4,Clerks,500,60,100
5,Secretaries,20,10,4
Total,Total,710,142,142


In [8]:
assert alloc["n_prop"].tolist() == [33, 17, 66, 215] and alloc["n_prop"].sum() == 331
assert sek.loc["Total", "n_proportionate"] == 142 == sek.loc["Total", "n_disproportionate"]
print("Allocations match the Stata do-file and Table 13.1.")

Allocations match the Stata do-file and Table 13.1.
